# Choosing the 8-bit windowSensor data is 12-bit. Saving it as 8-bit means picking a window: which pixelvalues become 0, which become 255, and what gets cut off at both ends.Two ways to pick it.| | how the window is found ||---|---|| **percentile** | from the pixels in front of you, every time || **fixed rule** | decided once, written down, applied everywhere |Four cases show where each one works and where it breaks.Red marks the pixels the window cuts off.

## Setup

In [ ]:
import sys, urllib.requestBASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'urllib.request.urlretrieve(f'{BASE}/lib/stretch.py', 'stretch.py')sys.modules.pop('stretch', None)from stretch import *urllib.request.urlretrieve(f'{BASE}/dataset/stretch/tiles.npz', 'tiles.npz')D = load('tiles.npz')

## Case 1 — an ordinary tileNothing extreme in the frame. Narrowing the percentile cuts more and uses moreof 0-255; widening it does the opposite.

In [ ]:
tile = D['urban']wins = windows(tile.mean(2).ravel(), [('pct', 0.1, 99.9), ('pct', 2, 98), ('pct', 5, 95)])show(tile, wins, xmax=1000)

## Case 2 — cloud in the poolSame tile. This time the window is read from a pool that also contains cloudfrom somewhere else (yellow in the histogram). The percentile windows followthe cloud; the fixed rule does not move.

In [ ]:
pool = np.concatenate([tile.mean(2).ravel(), D['cloud']])wins = windows(pool, [('pct', 0.1, 99.9), ('pct', 2, 98), ('rule',)])show(tile, wins, cloud=D['cloud'], xmax=1600)

## Case 3 — a fixed rule that fitsThe rule 150-800 was written down once. On this tile it lands close to where thepercentiles land, so it costs little.

In [ ]:
tile = D['lowrise']wins = windows(tile.mean(2).ravel(), [('pct', 0.1, 99.9), ('pct', 2, 98), ('rule',)])show(tile, wins, xmax=1000)

## Case 4 — the same rule somewhere elseA darker region. The percentiles follow the tile and still fill the range. Thesame fixed rule now sits above the data.

In [ ]:
tile = D['dark']wins = windows(tile.mean(2).ravel(), [('pct', 0.1, 99.9), ('pct', 2, 98), ('rule',)])show(tile, wins, xmax=1000)

## What to take away- A percentile window adapts to the frame — including to cloud and to anything  else bright that happens to be in it.- A fixed rule is stable and comparable across scenes, but only where it was derived.- Either way, check how much gets cut and how much of 0-255 is actually used.